# W1 Person B - Run LViT + UNet Evaluation

Notebook này chạy full-image validation evaluation cho cả LViT-T và UNet, lưu probability maps, entropy maps, overlap-disagreement maps, rồi sweep threshold `0.30 -> 0.90`.

Dùng validation để ra quyết định Week 1. Không dùng test để chọn threshold/model.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import pandas as pd
import torch
import yaml

RUN_LVIT = True
RUN_UNET = True
RUN_SWEEP = True
REPO_URL = 'https://github.com/lehngoc/BTXRD-LViT.git'
REPO_BRANCH = 'w1-person-b-eval'
CLONE_REPO_ON_KAGGLE = True
REQUIRE_T4 = True
MIN_CUDA_CAPABILITY = (7, 0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Device:', DEVICE)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print('GPU:', gpu_name)
    print('CUDA capability:', capability)
    if capability < MIN_CUDA_CAPABILITY:
        raise RuntimeError(
            f'GPU {gpu_name} has CUDA capability {capability}, but this PyTorch build needs >= {MIN_CUDA_CAPABILITY}. '
            'On Kaggle, switch Accelerator to GPU T4 or T4 x2, then restart the session.'
        )
    if REQUIRE_T4 and 'T4' not in gpu_name:
        raise RuntimeError(
            f'This notebook is configured for T4, but Kaggle assigned {gpu_name}. '
            'Change Session options -> Accelerator to GPU T4/T4 x2, then restart and rerun.'
        )
else:
    raise RuntimeError('No GPU found. On Kaggle, enable Accelerator = GPU T4/T4 x2 before running this notebook.')

## Locate Repo, Dataset, Artifacts

Trên Kaggle, notebook sẽ tìm dataset/artifact dưới `/kaggle/input`. Nếu chạy local, nó dùng repo hiện tại.

In [ ]:
def find_dir_with(relative_path, roots):
    rel = Path(relative_path)
    for root in roots:
        root = Path(root)
        if root.exists() and (root / rel).exists():
            return root
        if root.exists():
            for candidate in root.glob('**/*'):
                if candidate.is_dir() and (candidate / rel).exists():
                    return candidate
    return None

if CLONE_REPO_ON_KAGGLE and Path('/kaggle/working').exists():
    clone_target = Path('/kaggle/working/BTXRD-LViT')
    if clone_target.exists():
        shutil.rmtree(clone_target)
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(clone_target)], check=True)

cwd = Path.cwd()
repo_candidates = [cwd, cwd / 'BTXRD-LViT', Path('/kaggle/working/BTXRD-LViT')]
REPO_ROOT = next((p for p in repo_candidates if (p / 'src/inference/sliding_window.py').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Không thấy repo code. Hãy chạy notebook này từ repo BTXRD-LViT hoặc clone repo vào /kaggle/working/BTXRD-LViT.')

DATA_ROOT = find_dir_with('data/exports/btxrd_preprocessed/val.csv', [Path('/kaggle/input'), REPO_ROOT])
if DATA_ROOT is None:
    raise FileNotFoundError('Không thấy data/exports/btxrd_preprocessed/val.csv dưới /kaggle/input hoặc repo.')

UNET_ARTIFACT_DIR = find_dir_with('E2_unet_patch384_preprocessed_pos060/best.pt', [Path('/kaggle/input'), REPO_ROOT])
if UNET_ARTIFACT_DIR is not None:
    UNET_ARTIFACT_DIR = UNET_ARTIFACT_DIR / 'E2_unet_patch384_preprocessed_pos060'
else:
    UNET_ARTIFACT_DIR = find_dir_with('best.pt', [REPO_ROOT / 'E2_unet_patch384_preprocessed_pos060'])

LVIT_ARTIFACT_DIR = find_dir_with('E4_lvit_t_patch384_text_artifacts/best.pt', [Path('/kaggle/input'), REPO_ROOT])
if LVIT_ARTIFACT_DIR is not None:
    LVIT_ARTIFACT_DIR = LVIT_ARTIFACT_DIR / 'E4_lvit_t_patch384_text_artifacts'
else:
    LVIT_ARTIFACT_DIR = find_dir_with('best.pt', [REPO_ROOT / 'E4_lvit_t_patch384_text_artifacts'])

required_scripts = [
    REPO_ROOT / 'src/inference/evaluate_lvit_t_sliding_window.py',
    REPO_ROOT / 'src/inference/evaluate_unet_sliding_window.py',
    REPO_ROOT / 'src/inference/sweep_probability_maps.py',
]
missing_scripts = [str(p) for p in required_scripts if not p.exists()]
if missing_scripts:
    raise FileNotFoundError('Thiếu script W1 mới. Hãy upload/pull repo bản mới nhất. Missing:\n' + '\n'.join(missing_scripts))

print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('UNET_ARTIFACT_DIR:', UNET_ARTIFACT_DIR)
print('LVIT_ARTIFACT_DIR:', LVIT_ARTIFACT_DIR)

In [ ]:
required_data = [
    'data/exports/btxrd_preprocessed/val.csv',
    'data/processed/images_preprocessed',
    'data/processed/masks_preprocessed',
]
missing_data = [p for p in required_data if not (DATA_ROOT / p).exists()]
if missing_data:
    raise FileNotFoundError('Dataset thiếu:\n' + '\n'.join(missing_data))

if RUN_LVIT and (LVIT_ARTIFACT_DIR is None or not (LVIT_ARTIFACT_DIR / 'best.pt').exists()):
    raise FileNotFoundError('RUN_LVIT=True nhưng không thấy LViT best.pt')
if RUN_UNET and (UNET_ARTIFACT_DIR is None or not (UNET_ARTIFACT_DIR / 'best.pt').exists()):
    raise FileNotFoundError('RUN_UNET=True nhưng không thấy UNet best.pt')

val_df = pd.read_csv(DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv')
print('Val images:', len(val_df))
print(val_df[['image_id', 'tumor']].head())

## Create Runtime Configs

Config gốc trong repo có thể dùng `root_dir=.` hoặc path Kaggle cũ, nên cell này tạo config runtime đúng với input hiện tại.

In [ ]:
WORK_DIR = Path('/kaggle/working/w1_b_eval') if Path('/kaggle/working').exists() else REPO_ROOT / 'w1_b_eval'
WORK_DIR.mkdir(parents=True, exist_ok=True)

def write_runtime_config(base_config, output_path, batch_size=None, sw_batch_size=None):
    with open(base_config, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['data']['root_dir'] = str(DATA_ROOT)
    cfg['training']['device'] = DEVICE
    if batch_size is not None:
        cfg['training']['batch_size'] = batch_size
    if sw_batch_size is not None:
        cfg.setdefault('sliding_window', {})['batch_size'] = sw_batch_size
    cfg.setdefault('sliding_window', {})['patch_size'] = 384
    cfg.setdefault('sliding_window', {})['stride'] = 192
    cfg.setdefault('sliding_window', {})['merge'] = 'average_probability'
    with open(output_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return output_path

LVIT_CONFIG = write_runtime_config(
    REPO_ROOT / 'configs/train_lvit_t_patch384.yaml',
    WORK_DIR / 'w1_lvit_t_eval_runtime.yaml',
    batch_size=1,
    sw_batch_size=1,
)
UNET_CONFIG = write_runtime_config(
    REPO_ROOT / 'configs/train_unet_patch384.yaml',
    WORK_DIR / 'w1_unet_eval_runtime.yaml',
    sw_batch_size=4,
)
print('LVIT_CONFIG:', LVIT_CONFIG)
print('UNET_CONFIG:', UNET_CONFIG)
print('WORK_DIR:', WORK_DIR)

## Run LViT-T Validation

In [ ]:
def run(cmd):
    print('RUN:', ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True, cwd=str(REPO_ROOT))

if RUN_LVIT:
    lvit_prob_dir = WORK_DIR / 'w1_lvit_t_validation_probability_maps'
    run([
        sys.executable, REPO_ROOT / 'src/inference/evaluate_lvit_t_sliding_window.py',
        '--config', LVIT_CONFIG,
        '--checkpoint', LVIT_ARTIFACT_DIR / 'best.pt',
        '--split', 'val',
        '--device', DEVICE,
        '--output', WORK_DIR / 'w1_lvit_t_val_sliding_metrics_thr050.json',
        '--save-probability-dir', lvit_prob_dir,
        '--save-overlap-stats-dir', WORK_DIR / 'w1_lvit_t_overlap_disagreement_prototype',
        '--save-entropy-dir', WORK_DIR / 'w1_lvit_t_entropy_prototype',
        '--save-disagreement-dir', WORK_DIR / 'w1_lvit_t_overlap_disagreement_maps',
    ])
else:
    print('RUN_LVIT=False, skipped.')

## Sweep LViT-T Thresholds

In [ ]:
THRESHOLDS = '0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90'

if RUN_LVIT and RUN_SWEEP:
    run([
        sys.executable, REPO_ROOT / 'src/inference/sweep_probability_maps.py',
        '--manifest', DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
        '--root-dir', DATA_ROOT,
        '--probability-dir', WORK_DIR / 'w1_lvit_t_validation_probability_maps',
        '--output', WORK_DIR / 'w1_lvit_t_threshold_sweep.csv',
        '--per-image-output', WORK_DIR / 'w1_lvit_t_threshold_sweep_per_image.csv',
        '--thresholds', THRESHOLDS,
        '--min-fp-area-ratio', '0.001',
    ])
    display(pd.read_csv(WORK_DIR / 'w1_lvit_t_threshold_sweep.csv'))

## Run UNet Validation

In [ ]:
if RUN_UNET:
    run([
        sys.executable, REPO_ROOT / 'src/inference/evaluate_unet_sliding_window.py',
        '--config', UNET_CONFIG,
        '--checkpoint', UNET_ARTIFACT_DIR / 'best.pt',
        '--split', 'val',
        '--device', DEVICE,
        '--output', WORK_DIR / 'w1_unet_val_sliding_metrics_thr050.json',
        '--save-probability-dir', WORK_DIR / 'w1_unet_validation_probability_maps',
        '--save-overlap-stats-dir', WORK_DIR / 'w1_unet_overlap_disagreement_prototype',
        '--save-entropy-dir', WORK_DIR / 'w1_unet_entropy_prototype',
        '--save-disagreement-dir', WORK_DIR / 'w1_unet_overlap_disagreement_maps',
    ])
else:
    print('RUN_UNET=False, skipped.')

## Sweep UNet Thresholds

In [ ]:
if RUN_UNET and RUN_SWEEP:
    run([
        sys.executable, REPO_ROOT / 'src/inference/sweep_probability_maps.py',
        '--manifest', DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
        '--root-dir', DATA_ROOT,
        '--probability-dir', WORK_DIR / 'w1_unet_validation_probability_maps',
        '--output', WORK_DIR / 'w1_unet_threshold_sweep.csv',
        '--per-image-output', WORK_DIR / 'w1_unet_threshold_sweep_per_image.csv',
        '--thresholds', THRESHOLDS,
        '--min-fp-area-ratio', '0.001',
    ])
    display(pd.read_csv(WORK_DIR / 'w1_unet_threshold_sweep.csv'))

## Compare Summary

In [ ]:
summary_rows = []
for model_name, path in [
    ('LViT-T', WORK_DIR / 'w1_lvit_t_threshold_sweep.csv'),
    ('UNet', WORK_DIR / 'w1_unet_threshold_sweep.csv'),
]:
    if path.exists():
        df = pd.read_csv(path)
        best_dice = df.sort_values('tumor_dice', ascending=False).iloc[0].to_dict()
        best_low_fp = df.sort_values(['normal_fp_image_rate', 'tumor_dice'], ascending=[True, False]).iloc[0].to_dict()
        summary_rows.append({'model': model_name, 'selection': 'best_val_tumor_dice', **best_dice})
        summary_rows.append({'model': model_name, 'selection': 'lowest_normal_fp_then_dice', **best_low_fp})

summary = pd.DataFrame(summary_rows)
summary_path = WORK_DIR / 'w1_lvit_unet_threshold_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
display(summary)

## Package W1 Outputs

In [ ]:
zip_path = Path('/kaggle/working/w1_b_eval_outputs.zip') if Path('/kaggle/working').exists() else REPO_ROOT / 'w1_b_eval_outputs.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', WORK_DIR)
print('Packaged:', zip_path)
print('Output dir:', WORK_DIR)